In [12]:
from math import sin, cos

def f(a, b, c): # This is the expression we will pass 
  return -a**3 + sin(3*b) - 1.0/c + b**2.5 - a**0.5

print(f(2, 3, 4))


6.336362190988558


In [13]:
def gradf(a, b, c):
  """
  Get and return the derivative
  """
  da = (-3*a**2) + (0.5*-a**-0.5)
  db = (3*cos(3*b)) + (2.5*b**1.5)
  dc = -(-1/c**2)
  return [da, db, dc]

#Test
ans = [-12.353553390593273, 10.25699027111255, 0.0625] # Expected derivatives
yours = gradf(2, 3, 4)
for dim in range(3): 
  ok = 'OK' if abs(yours[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {yours[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353553390593273
OK for dim 1: expected 10.25699027111255, yours returns 10.25699027111255
OK for dim 2: expected 0.0625, yours returns 0.0625


In [14]:
# Getting the derivative using the limit formula
h = 0.00000001
dfda = (f(2+h, 3, 4) - f(2, 3, 4))/h #Remember, f(x) = -a**3 + sin(3*b) - 1.0/c + b**2.5 - a**0.5 | so, f(a+h,b,c) = f(x + h)
dfdb = (f(2, 3+h, 4) - f(2, 3, 4))/h
dfdc = (f(2, 3, 4+h) - f(2, 3, 4))/h
numerical_grad = [dfda, dfdb, dfdc]

for dim in range(3):
  ok = 'OK' if abs(numerical_grad[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353553380251014
OK for dim 1: expected 10.25699027111255, yours returns 10.256990368162633
OK for dim 2: expected 0.0625, yours returns 0.0624999607623522


In [15]:
# Getting the derivative using central difference formula | f(x+h) - f(x-h)/2h 
sdfda = (f(2+h, 3, 4) - f(2-h, 3, 4))/(2*h)
sdfdb = (f(2, 3+h, 4) - f(2, 3-h, 4))/(2*h)
sdfdc = (f(2, 3, 4+h) - f(2, 3, 4-h))/(2*h)
numerical_grad2 = [sdfda, sdfdb, sdfdc] 

for dim in range(3):
  ok = 'OK' if abs(numerical_grad2[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad2[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353553291433172
OK for dim 1: expected 10.25699027111255, yours returns 10.256990368162633
OK for dim 2: expected 0.0625, yours returns 0.0624999607623522


Softmax

In [16]:
from math import exp, log

class Value:

  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None # Returns none because gradients should start at 0 when computing derivatives
    self._prev = set(_children)
    self._op = _op
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data})"

  def __add__(self, other): 
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')

    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward

    return out

  # Multiplication
  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self,other), '*')

    def _backward():
      other.grad += self.data * out.grad
      self.grad += other.data * out.grad
    out._backward = _backward

    return out
  
  #Euler
  def exp(self):
    out = Value(exp(self.data), (self, ), "exp")
    def _backward():
      self.grad += out.data * out.grad
    out._backward = _backward
    return out
  
  #Logarithm
  def log(self):
    out = Value(log(self.data), (self, ), "log")
    
    def _backward():
      self.grad += (1/self.data) * out.grad
      
    out._backward = _backward
    return out
  
  # Power
  def __pow__(self, other):
    assert isinstance(other, (int, float))
    out = Value(self.data**other, (self, ), "^")

    def _backward():
      self.grad += (other*(self.data**(other-1))) * out.grad
    out._backward = _backward
    return out
  
  # Sin
  def sin(self):
    out = Value(sin(self.data), (self, ), "sin")
  
    def _backward():
      self.grad += (cos(self.data)) * out.grad
    
    out._backward = _backward
    return out
  
  #Tanh
  def tanh(self):
    e2x = (self * 2).exp()
    return (e2x - 1) / (e2x + 1)
  
  def __sub__(self, other):
    return self + (-other)
  def __radd__(self, other):
    return self + other
  def __rmul__(self, other):
    return self * other
  def __rsub__(self, other):
    return self - other
  def __neg__(self):
    return self * -1
  def __truediv__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    return self *(other**-1)
  def __rtruediv__(self, other): 
    return Value(other) * (self**-1)

  def backward(self): 
    topo = [] #Creates an empty list
    visited = set() #Prevents nodes from repeating
    def build_topo(v):
      if v not in visited:
        visited.add(v) #Append to the set so that it does not repeat
        for child in v._prev: #Takes the values that amount to output v
          build_topo(child) #Repeat until every child is accounted for
        topo.append(v) #Append the first child before the parent
    build_topo(self) 

    self.grad = 1.0 #Derivative of final output
    for node in reversed(topo): #Reversed because we want to start from the top
      node._backward() #Calls the backward pass which is the function to get the gradient of the out value

In [17]:
# Negative log likelihood loss, oftentimes used in classifications
# this is the softmax function
# https://en.wikipedia.org/wiki/Softmax_function

def softmax(logits):
  counts = [logit.exp() for logit in logits] #Creates a list of euler values
  denominator = sum(counts) 
  out = [c / denominator for c in counts] #A list of quotients 
  return out

logits = [Value(0.0), Value(3.0), Value(-2.0), Value(1.0)]
probs = softmax(logits)
loss = -probs[3].log() # dim 3 acts as the label for this input example
print(loss.data)

ans = [0.041772570515350445, 0.8390245074625319, 0.005653302662216329, -0.8864503806400986]
loss.backward()
for dim in range(4):
  ok = 'OK' if abs(logits[dim].grad - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {logits[dim].grad}")

2.1755153626167147
OK for dim 0: expected 0.041772570515350445, yours returns 0.041772570515350445
OK for dim 1: expected 0.8390245074625319, yours returns 0.8390245074625319
OK for dim 2: expected 0.005653302662216329, yours returns 0.005653302662216329
OK for dim 3: expected -0.8864503806400986, yours returns -0.8864503806400986


In [18]:
import random
import torch

In [19]:
class Module:
    def zero_grad(self, params):
        for p in params:
            p.grad = 0.0
    
    def parameters(self):
        return []

class Neuron(Module):
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))
        
    def __call__(self, x):
        act = sum( wi*xi for wi, xi in zip(self.w, x)) + self.b
        out = act.tanh()
        return out
    
    def parameters(self):
        return self.w + [self.b]
    
class Layer(Module):
    def __init__(self,nin,nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
        
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]
    
class MLP(Module):
    def __init__(self,nin,nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

n = MLP(3,[4,4,1])
x = [Value(0.0), Value(3.0), Value(-2.0), Value(1.0)]

n(x)


Value(data=0.6144144113472001)

In [ ]:
xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]

n = MLP(3, [4, 4, 1])
# 19 7.991587261068943
for k in range(20):
    # forward pass
    ypred = [n(x) for x in xs]
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))

    # backward pass
    n.zero_grad(n.parameters())
    loss.backward()

    # update
    for p in n.parameters():
        p.data += -0.01 * p.grad

    print(k, loss.data)

0 3.2011993631523294
1 0.8605967483019954
2 4.001114259188741
3 3.9999937630930265
4 3.9999926077250416
5 3.9999915655106504
6 3.9999906192221593
7 3.999989755008932
8 3.999988961600755
9 3.9999882297279545
10 3.9999875516921426
11 3.999986921043635
12 3.9999863323356983
13 3.9999857809349963
14 3.9999852628737367
15 3.999984774733126
16 3.9999843135506534
17 3.999983876745656
18 3.9999834620590855
19 3.9999830675044104


In [21]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
c = torch.tensor(4.0, requires_grad=True)

L = -a**3 + torch.sin(3*b) - 1.0/c + b**2.5 - a**0.5
L.backward()

print(a.grad.item(), b.grad.item(), c.grad.item())


-12.353553771972656 10.256989479064941 0.0625


In [22]:
#Draw_dot
from graphviz import Digraph

def trace(root):
    nodes, edges = set(), set()
    
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes,edges 
    
def draw_dot(root):
    dot = Digraph(format='svg',graph_attr={'rankdir':'LR'})
    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        dot.node(name=uid,label = "{ %s | data %.6f | grad %.2f }" % (n.label, n.data, n.grad),shape='record')
        if n._op:
            dot.node(name=uid+n._op,label=n._op)
            dot.edge(uid+n._op,uid)
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2))  + n2._op)
        
    return dot


draw_dot(L)

AttributeError: 'Tensor' object has no attribute '_prev'

In [ ]:
draw_dot(f(2, 3, 4))



AttributeError: 'float' object has no attribute '_prev'

In [ ]:
a = Value(2.0)
b = Value(3.0)
c = Value(4.0)

#L1 = -a**3 + sin(3*b) - 1.0/c + b**2.5 - a**0.5

In [ ]:
L1 = -a**3 + (3*b).sin() - 1.0/c + b**2.5 - a**0.5

In [ ]:
draw_dot(L1)

ExecutableNotFound: failed to execute PosixPath('dot'), make sure the Graphviz executables are on your systems' PATH